<a href="https://colab.research.google.com/github/taavip/DL4CV/blob/main/HW3_updated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Homework #3**
## Vision Transformers

This colaboratory contains Homework #3 of the Deep Learning for Computer Vision course, which is due **April 19, Sunday 2026, midnight (23:59 EET time)**. To complete the homework, extract **(File -> Download .ipynb)** and submit to the course webpage.

**NB! Links to your colaboratory will not be accepted as a solution!**

## Submission's rules:

1. Please, submit only .ipynb that you extract from the Colaboratory.
2. Run your homework exercises before submitting (output should be present, preferably restart the kernel and press run all the cells).
3. Do not change the description of tasks in red (even if there is a typo|mistake|etc).
4. Please, make sure to avoid unnecessary long printouts.
5. Each task should be solved right under the question of the task and not elsewhere.
6. Solutions to both regular and bonus exercises should be submitted in one IPYNB file.

Please, steer clear of copying someone else's work. If you discuss assignments with anyone in the course, please, mention their names here:
1. Pooh

## List of Homework's exercises:
1. [EX1](#scrollTo=Qva9-9bIc8OL&uniqifier=1) - 5 points
2. [EX2](#scrollTo=6KSejfW0c8Og&uniqifier=1) - 8 points (+ up to 3 bonus points)
3. [EX3](#scrollTo=rJv0SN_ic8Oo&uniqifier=1) - 5 points


<a id='exercise_1'></a>
# **Exercise 1 (5 points): Back to Vision Transformer and Multi-Head Attention**

<font color='red'> In the first exercise of your homework, you will take a detailed look at a ground-up implementation of the famous Vision Transformer (ViT) model. We examined one during the practice session for a classification task. Here, you will practice implementing ViT from the beginning. Additionally, we previously used a Torch implementation of Multi-Head Attention (e.g., `nn.MultiheadAttention`). In this homework, we will implement one ourselves and compare its performance.  

In [ ]:
import os
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data
import torchvision

import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from tqdm.notebook import tqdm_notebook

from tqdm import tqdm

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torchvision.datasets import VOCSegmentation
from torch.utils.data import DataLoader, Dataset
import timm

In [ ]:
random.seed(3407)

device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print(device)

<font color='red'> Let's first start by loading the MNIST dataset, as we did previously. We will be using it to train our Transformer classifier.  

In [ ]:
transforms = torchvision.transforms.Compose([torchvision.transforms.ToTensor(),
                                             torchvision.transforms.Normalize((0.5,), (0.5,))])

# download MNIST training set and reserve 50000 for training
train_dataset = torchvision.datasets.MNIST(root='./data/MNIST', train=True,
                                           download=True, transform=transforms)
train_set, val_set = torch.utils.data.random_split(train_dataset, [50000, 10000])

# download MNIST test set
test_set = torchvision.datasets.MNIST(root='./data/MNIST', train=False,
                                      download=True, transform=transforms)


# We define the data loaders using the datasets
train_loader = torch.utils.data.DataLoader(dataset=train_set, batch_size=32, shuffle=True)
val_loader = torch.utils.data.DataLoader(dataset=val_set, batch_size=32, shuffle=False)
test_loader = torch.utils.data.DataLoader(dataset=test_set, batch_size=32, shuffle=False)

<font color='red'>**(Homework exercise 1 – a)** In the first exercise we will be implementing the Vision Tranformer model we have look into on our practice session. You can refer to it using this link.
Implement the division on an image into patches.
(0.5 points)</font>

<font color='red'> Remember that in ViT, we start by dividing our image into 16x16 patches. We can implement this using the `torch.reshape()` function. First, we divide our spatial dimensions by the patch size, and then we merge everything into the channel dimension. The resulting tensor should have the shape `(b, l, d)`, where `l` is the number of patches in our image and `d` is the dimension of each patch. This dimension effectively consists of all flattened pixels in a given patch and the original channel dimension of our image.  

<font color='red'> Define a patch size and implement the patchification process below.  

In [ ]:
#### YOUR CODE STARTS HERE ####

x = torch.randn(1, 3, 64, 64)
patch_size = ...

B, C, H, W = x.shape
x = x.reshape(...)
#### YOUR CODE ENDS HERE ####

print(x.shape)

# expect to get a tensor of shape (1, 16, 768)
assert x.shape == torch.Size([1, 16, 768])

<font color='red'>**(Homework exercise 1 – b)** Implement InputEmbedding, EncoderBlock, and VitTransformer (2.5 points)</font>

<font color='red'> Now, we can implement a wrapper function called `ImageEmbedding`, which will divide our input image into patches, apply an `nn.Linear` projection on them, and return patch embeddings (tokens). Since we are implementing a Vision Transformer for classification, we also need a `[cls]` token, which we concatenate to our sequence. In `ImageEmbedding`, we additionally add learnable positional embeddings for each token in our sequence.  

<font color='red'> Below, implement the required functionality for the `ImageEmbedding` class. The `get_patches()` implementation should be the same as the one you defined above.  


In [ ]:
class InputEmbedding(nn.Module):
	def __init__(self, patch_size, n_channels, embed_dim, num_patches):
		super(InputEmbedding, self).__init__()

        #### YOUR CODE STARTS HERE ####
		self.patch_size = patch_size
		self.n_channels = n_channels
		self.embed_dim = embed_dim

		self.proj = ...
		self.class_token = ...
		self.pos_embed = ...
        #### YOUR CODE ENDS HERE ####

	def get_patches(self, x):
        #### YOUR CODE STARTS HERE ####
		B, C, H, W = x.shape
        # your patching goes here
		...
        #### YOUR CODE ENDS HERE ####

		return x

	def forward(self, x):
		batch_size = x.shape[0]

        #### YOUR CODE STARTS HERE ####
		# (b, c, h, w) -> (b, c, num_patches, patch_size * patch_size * c)
		# num_patches = (h // patch_size) ** 2
		patches = ...

		x = ...
		class_token = ...

		x = torch.cat((class_token, x), dim=1)

        #### YOUR CODE ENDS HERE ####

		return x

<font color='red'> The `EncoderBlock` will encapsulate the attention part as well as the feed-forward (MLP) part. Implement the `EncoderBlock` functionality. For `self.attn`, use the `torch.nn` implementation.  

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.0):
        """
        Inputs:
            embed_dim - Dimensionality of input and attention feature vectors
            num_heads - Number of heads to use in the Multi-Head Attention block
            dropout - Amount of dropout to apply in the feed-forward network
        """
        super().__init__()

        #### YOUR CODE STARTS HERE ####
        self.norm1 = ...
        self.attn = ...
        self.norm2 = ...
        self.proj = ...
        #### YOUR CODE ENDS HERE ####

    def forward(self, x):
        #### YOUR CODE STARTS HERE ####
        ...
        #### YOUR CODE ENDS HERE ####

        return x

<font color='red'> Now, let's put everything together! Given an image, we will first use `self.embedding` to get our tokens. After that, we should apply our `self.encoder` blocks to refine the tokens and share information with our (initially) random `[cls]` token. Finally, to get class probabilities, we need to apply an MLP head to our `[cls]` token.  

<font color='red'> Thus, after the encoder blocks, retrieve the `[cls]` token and feed it to the MLP layer.  

<font color='red'> **Q:** Do we need to add a `softmax()` activation at the end (yes/no?) and why?  

In [ ]:
class VitTransformer(nn.Module):
    def __init__(
        self,
        embed_dim,
        num_channels,
        num_heads,
        num_layers,
        num_classes,
        patch_size,
        img_size,
        dropout=0.0,
    ):
        """
        Inputs:
            embed_dim - Dimensionality of the input feature vectors to the Transformer
            hidden_dim - Dimensionality of the hidden layer in the feed-forward networks
                         within the Transformer
            num_channels - Number of channels of the input (3 for RGB or 1 for grayscale)
            num_layers - Number of layers to use in the Transformer
            num_classes - Number of classes to predict
            patch_size - Number of pixels that the patches have per dimension
            img_size - Image size
            dropout - Amount of dropout to apply in the feed-forward network and
                      on the input encoding
        """
        super().__init__()

        num_patches = (img_size // patch_size) ** 2

        #### YOUR CODE STARTS HERE ####
        self.embedding = ...

        self.encoders = nn.ModuleList([
            ...
            for _ in range(num_layers)
        ])

        self.mlp_head = ...
        self.dropout = ...
        #### YOUR CODE ENDS HERE ####


    def forward(self, x):
        #### YOUR CODE STARTS HERE ####
        x = self.embedding(x)
        # tip: look into batch_first for nn.MultiHeadAttention and consider the dimentions order for our implementation

        ...

        cls = ...
        out = ...
        #### YOUR CODE ENDS HERE ####

        return out

<font color='red'> **Your answer goes here:**

(question about softmax())

...

<font color='red'> Let's now test our implementation of ViT. Define meaningful parameters for ViT, which we will train on the MNIST dataset.  

<font color='red'> Explain the main parameters and your choice behind them.  

In [ ]:
#### YOUR CODE STARTS HERE ####
image_size = ...
patch_size = ...
num_channels = ...
embed_dim = ...
num_heads = ...
num_layers = ...
num_classes = ...
dropout = ...
batch_size = ...
epochs = ...
base_lr = ...
weight_decay = ...
#### YOUR CODE ENDS HERE ####

<font color='red'> **Your answer goes here:**

...

In [ ]:
model = VitTransformer(
    img_size=image_size,
    embed_dim=embed_dim,
    num_heads=num_heads,
    num_layers=num_layers,
    patch_size=patch_size,
    num_channels=num_channels,
    num_classes=num_classes,
    dropout=dropout
).to(device)

In [ ]:
x = torch.randn(batch_size, num_channels, image_size, image_size).to(device)
out = model(x)
print(x.shape)
print(out.shape)

In [ ]:
# here is a sanity check for an output shape
assert out.shape == torch.Size([batch_size, num_classes])

<font color='red'>**(Homework exercise 1 – c)** For the second part implemnt the Trainer which will take the job of training and evaluating your ViT classification model. (2 points)</font>

<font color='red'> Now, we will implement another important component to train our Vision Transformer model. Sometimes, it is useful not only to have a training loop but also to encapsulate everything in a single instance to better track experiments when running the model multiple times with different configurations.  

<font color='red'> Here, we implement a simple general `Trainer()` instance. This trainer will receive the required parameters as input and will handle training inside the `Trainer.train()` function. The `train` function will run a loop over the epochs, with a training step and a validation step over the entire set of batches.  

<font color='red'> Additionally, the `Trainer` will implement a `.test()` function to evaluate the performance of our model (`self.model`).  

<font color='red'> Implement the functionality for `train_step()`, `valid_step()`, and `test()`.  

In [ ]:
class Trainer:
    def __init__(self,
                 max_epochs,
                 model,
                 criterion,
                 optimizer,
                 scheduler,
                 device,
                 train_loader,
                 valid_loader,
                 test_loader,
				 log_every_n_steps=50
    ) -> None:
        self.model = model
        self.criterion = criterion
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.device = device
        self.train_loader = train_loader
        self.valid_loader = valid_loader
        self.test_loader = test_loader
        self.max_epochs = max_epochs
        self.log_every_n_steps = log_every_n_steps


    def train_step(self):
        self.model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        pbar = tqdm(self.train_loader, desc="Training")

        for batch_idx, (inputs, targets) in enumerate(pbar):
            inputs, targets = inputs.to(self.device), targets.to(self.device)

            #### YOUR CODE STARTS HERE ####
            ...
            #### YOUR CODE ENDS HERE ####

            running_loss += loss.item()

            #### YOUR CODE STARTS HERE ####
            _, predicted = ...
            correct += ...
            total += targets.size(0)

            if batch_idx % self.log_every_n_steps == 0:
                avg_loss = ...
                accuracy = ...
                pbar.set_postfix(loss=f"{avg_loss:.4f}", acc=f"{accuracy:.2f}%")
            #### YOUR CODE ENDS HERE ####

        avg_loss = running_loss / len(self.train_loader)
        accuracy = 100 * correct / total
        pbar.set_postfix(loss=f"{avg_loss:.4f}", acc=f"{accuracy:.2f}%")
        return avg_loss, accuracy


    def valid_step(self):
        self.model.eval()
        running_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            pbar = tqdm(self.valid_loader, desc="Validation")

            for batch_idx, (inputs, targets) in enumerate(pbar):
                inputs, targets = inputs.to(self.device), targets.to(self.device)

                #### YOUR CODE STARTS HERE ####
                ...
                #### YOUR CODE ENDS HERE ####

                running_loss += loss.item()

                #### YOUR CODE STARTS HERE ####
                _, predicted = ...
                correct += ...
                total += targets.size(0)

                if batch_idx % self.log_every_n_steps == 0:
                    avg_loss = ...
                    accuracy = ...
                    pbar.set_postfix(loss=f"{avg_loss:.4f}", acc=f"{accuracy:.2f}%")
                #### YOUR CODE ENDS HERE ####

            avg_loss = running_loss / len(self.valid_loader)
            accuracy = 100 * correct / total
            pbar.set_postfix(loss=f"{avg_loss:.4f}", acc=f"{accuracy:.2f}%")

        return avg_loss, accuracy

    def train(self):
        self.model.to(self.device)
        train_losses = []
        train_accuracies = []
        valid_losses = []
        valid_accuracies = []

        for epoch in range(self.max_epochs):
            train_loss, train_acc = self.train_step()
            valid_loss, valid_acc = self.valid_step()

            self.scheduler.step()

            train_losses.append(train_loss)
            train_accuracies.append(train_acc)
            valid_losses.append(valid_loss)
            valid_accuracies.append(valid_acc)

            print(f"Epoch {epoch+1}/{epochs}: train_loss: {train_loss:.4f}, train_acc: {train_acc:.2f}%, valid_loss: {valid_loss:.4f}, valid_acc: {valid_acc:.2f}%")
            print()

        return train_losses, train_accuracies, valid_losses, valid_accuracies

    def test(self):
        self.model.eval()
        correct = 0
        total = 0

        with torch.no_grad():
            pbar = tqdm(self.test_loader, desc="Testing")

            for inputs, targets in pbar:
                #### YOUR CODE STARTS HERE ####
                inputs, targets = inputs.to(self.device), targets.to(self.device)

                ...

                _, predicted = ...
                correct += ...
                total += targets.size(0)
                #### YOUR CODE ENDS HERE ####

            accuracy = 100 * correct / total
            print(f"test_acc: {accuracy:.2f}%")

        return accuracy

    def plot_metrics(self, train_losses, train_accuracies, valid_losses, valid_accuracies):
        fig, ax = plt.subplots(1, 2, figsize=(15, 5))
        ax[0].plot(train_losses, label="train_loss", color="blue")
        ax[0].plot(valid_losses, label="valid_loss", linestyle="--", color="orange")
        ax[0].set_title("Losses")

        ax[1].plot(train_accuracies, label="train_acc", color="blue")
        ax[1].plot(valid_accuracies, label="valid_acc", linestyle="--", color="orange")
        ax[1].set_title("Accuracies")

        for i in range(2):
            ax[i].legend()
            ax[i].grid(0.35)
            ax[i].set_xlabel("Epochs")

        plt.show()

<font color='red'> Now, for the task of classifying MNIST images into numbers, define our optimizer, loss function, and scheduler.  

<font color='red'> *(Tip: Make sure the scheduler step is placed correctly in the Trainer; different schedulers work with different steps.)*  

<font color='red'> Also, define your `Trainer` with a sufficiently large number of epochs.  

In [ ]:
#### YOUR CODE STARTS HERE ####
optimizer = ...
criterion = ...
scheduler = ...

trainer = Trainer(
    max_epochs=...,
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    train_loader=train_loader,
    valid_loader=val_loader,
    test_loader=test_loader
)
#### YOUR CODE ENDS HERE ####

<font color='red'> Now lets run our training!!! Your should expect to get valid_acc ~96%

In [ ]:
train_losses, train_accuracies, valid_losses, valid_accuracies = trainer.train()

In [ ]:
train_losses

In [ ]:
trainer.plot_metrics(train_losses, train_accuracies, valid_losses, valid_accuracies)

In [ ]:
trainer.test()

<font color='red'> How good is our model?

<font color='red'> **Your answer goes here:**

...

# **Multi-Head Attention**

<font color='red'>**(Homework exercise 1 – d)** Implement your own MultiHeadAttention() and test its' performance. (2 points)</font>

<font color='red'> Now, we will implement the big boss. We initially touched on this part in the second practice session on Transformers, where we implemented the Pyramid Vision Transformer. Now, we need to modify that code to a basic state to work with our Vision Transformer.  

<font color='red'> First, define a single projection for our `q`, `k`, and `v`.  

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, dim, num_heads=8, qkv_bias=False, qk_scale=None, attn_drop=0., proj_drop=0.):
        super().__init__()
        assert dim % num_heads == 0, f"dim {dim} should be divided by num_heads {num_heads}."

        #### YOUR CODE STARTS HERE ####
        self.dim = dim
        self.num_heads = num_heads
        head_dim = ...
        self.scale = qk_scale or head_dim ** -0.5

        self.qkv = ...
        self.attn_drop = ...
        self.proj = ...
        self.proj_drop = ...
        #### YOUR CODE ENDS HERE ####


    def forward(self, x):
        B, N, C = x.shape
        #### YOUR CODE STARTS HERE ####
        qkv = self.qkv(x)
        ...
        q, k, v = qkv[0], qkv[1], qkv[2]

        attn = ...

        x = ...
        #### YOUR CODE ENDS HERE ####

        return x

In [ ]:
b, l, d = 1, 64, 256

x = torch.randn(b, l, d)
attn = MultiHeadAttention(d)

out = attn(x)
print(out.shape)

In [ ]:
# here is a sanity check for an output shape
assert out.shape == torch.Size([1, 64, 256])

<font color='red'> **Q1:** Can you explain the intuition behind using `self.attn_drop`? Why do you think we need it, and how can it help?  

<font color='red'> **Q2:** After projecting our input `x` into a `qkv` vector, we need to properly reshape it to correctly extract separate `q`, `k`, and `v` vectors. Explain the reshaping procedure over tensors using the following notation:  

- <font color='red'>`b` - batch size  
- `l` - sequence length  
- `d` - dimension of each element (token) in our sequence  
- `head_dim` - dimension of each head vector  
- `num_heads` - number of heads in the multi-head attention module  

<font color='red'> Write down the steps that transform the `qkv` vector from shape `(b, ???, d)` to separate vectors `q`, `k`, and `v`, each with shape `(b, num_heads, l, head_dim)`.  

<font color='red'>Start with:
- <font color='red'> qkv (b, ???, d) -> (b, l, ???, num_heads, head_dim)
- ...

<font color='red'> **Your answer goes here:**

...

<font color='red'> Now, let's define our second version of the Transformer Encoder Block, which uses our implementation of Multi-Head Attention.  


In [ ]:
class EncoderBlock_v2(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.0):
        """
        Inputs:
            embed_dim - Dimensionality of input and attention feature vectors
            num_heads - Number of heads to use in the Multi-Head Attention block
            dropout - Amount of dropout to apply in the feed-forward network
        """
        super().__init__()

        #### YOUR CODE STARTS HERE ####
        self.norm1 = ...
        self.attn = ...
        self.norm2 = ...
        self.proj = ...
        #### YOUR CODE ENDS HERE ####

    def forward(self, x):
        #### YOUR CODE STARTS HERE ####
        ...
        #### YOUR CODE ENDS HERE ####

        return x

In [ ]:
b, l, d = 1, 64, 256

x = torch.randn(b, l, d)
encoder = EncoderBlock_v2(d, 8)

out = encoder(x)
print(out.shape)

In [ ]:
# here is a sanity check for an output shape
assert out.shape == torch.Size([1, 64, 256])

In [ ]:
class VitTransformer_v2(nn.Module):
    def __init__(
        self,
        embed_dim,
        num_channels,
        num_heads,
        num_layers,
        num_classes,
        patch_size,
        img_size,
        dropout=0.0,
    ):
        """
        Inputs:
            embed_dim - Dimensionality of the input feature vectors to the Transformer
            hidden_dim - Dimensionality of the hidden layer in the feed-forward networks
                         within the Transformer
            num_channels - Number of channels of the input (3 for RGB or 1 for grayscale)
            num_layers - Number of layers to use in the Transformer
            num_classes - Number of classes to predict
            patch_size - Number of pixels that the patches have per dimension
            img_size - Image size
            dropout - Amount of dropout to apply in the feed-forward network and
                      on the input encoding
        """
        super().__init__()

        num_patches = (img_size // patch_size) ** 2

        #### YOUR CODE STARTS HERE ####
        self.embedding = ...

        self.encoders = nn.ModuleList([
            ...
            for _ in range(num_layers)
        ])

        self.mlp_head = ...
        self.dropout = ...
        #### YOUR CODE ENDS HERE ####


    def forward(self, x):
        #### YOUR CODE STARTS HERE ####
        x = self.embedding(x)
        # tip: look into batch_first for nn.MultiHeadAttention and consider the dimentions order for our implementation

        ...

        cls = ...
        out = ...
        #### YOUR CODE ENDS HERE ####

        return out

In [ ]:
model_v2 = VitTransformer_v2(
    img_size=image_size,
    embed_dim=embed_dim,
    num_heads=num_heads,
    num_layers=num_layers,
    patch_size=patch_size,
    num_channels=num_channels,
    num_classes=num_classes,
    dropout=dropout
).to(device)

In [ ]:
x = torch.randn(batch_size, num_channels, image_size, image_size).to(device)
out = model_v2(x)
print(x.shape)
print(out.shape)

In [ ]:
# here is a sanity check for an output shape
assert out.shape == torch.Size([batch_size, num_classes])

<font color='red'> Define the same training configuration as in the previous training and train the model.  

In [ ]:
#### YOUR CODE STARTS HERE ####
optimizer = ...
criterion = ...
scheduler = ...


trainer = Trainer(
    max_epochs=...,
    model=model_v2,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    train_loader=train_loader,
    valid_loader=val_loader,
    test_loader=test_loader
)
#### YOUR CODE ENDS HERE ####

In [ ]:
train_losses, train_accuracies, valid_losses, valid_accuracies = trainer.train()

In [ ]:
trainer.plot_metrics(train_losses, train_accuracies, valid_losses, valid_accuracies)

In [ ]:
trainer.test()

<font color='red'> If implemented correctly you should expect to get similar classification performance on validation set as you did with torch version of nn.MultiHeadAttention()

<font color='red'> **Your answer goes here:**

...

# **Exercise 2: SWIN Transformer as a General Backbone**

<font color='red'> In this task, we will look into the **Swin Transformer**. You can read more about it here: [Swin Transformer Paper](https://arxiv.org/abs/2103.14030).  

<font color='red'> The **Swin Transformer** (Shifted Window Transformer) is a hierarchical vision transformer model designed to process images efficiently while maintaining global context. Unlike traditional transformers, which rely on full self-attention (making them computationally expensive for high-resolution images), the Swin Transformer introduces **shifted windows** to limit self-attention to local regions while still allowing information exchange across the entire image.  

<font color='red'> The key components of the Swin Transformer are:  
- <font color='red'>**Hierarchical feature representation**: The model progressively reduces spatial dimensions while increasing feature richness, similar to CNNs.  
- **Shifted window-based self-attention**: Instead of computing global attention, Swin Transformer restricts attention to non-overlapping windows and periodically shifts these windows to capture spatial relationships.  
- **Patch merging**: A mechanism to downsample feature maps and increase receptive field, analogous to pooling layers in CNNs.  

<font color='red'> This architecture has been widely used in **image classification, object detection, and segmentation tasks** due to its efficiency and effectiveness.

<font color='red'>**(Homework exercise 2 – a)** Lets try using timm library to load a pretrained model, you can read more about timm models here: (https://huggingface.co/docs/timm/en/feature_extraction). (1 points)</font>

<font color='red'> Let's start by loading a **pretrained ResNet-50** architecture for extracting multi-scale features. We need to set `features_only=True` so that the model returns a list of feature maps at different resolutions.  

<font color='red'> You can follow a specific example as done here:  
https://huggingface.co/docs/timm/en/feature_extraction#create-a-feature-map-extraction-model  

<font color='red'> After loading the backbone, **print out the shapes of the extracted features** to understand how feature dimensions change across different layers. This helps us analyze how spatial resolution decreases while the number of channels increases in deeper layers, which is a characteristic of convolutional networks.

In [ ]:
#### YOUR CODE STARTS HERE ####
backbone = ...

x = torch.randn(8, 3, 224, 224)
out = backbone(x)

# print the feature shapes
...
#### YOUR CODE ENDS HERE ####

<font color='red'> Expected feature map sizes for **ResNet-50**  

<font color='red'> The extracted feature maps follow the format **(b, c, h, w)**, where:  
- `b` is the batch size  
- `c` is the number of channels  
- `h, w` are the spatial dimensions
</font>

```
torch.Size([8, 64, 112, 112])  
torch.Size([8, 256, 56, 56])  
torch.Size([8, 512, 28, 28])  
torch.Size([8, 1024, 14, 14])  
torch.Size([8, 2048, 7, 7])  
```

<font color='red'> For such a **ResNet-50** model, we define:  

```python
embed_dims = [64, 256, 512, 1024, 2048]
```

<font color='red'> These values represent the **channel dimensions of feature maps** extracted at different stages of the network. The early layers capture low-level details with a higher resolution, while deeper layers capture more abstract and complex patterns at a lower resolution.  

<font color='red'> Also lets load the **swin small** model for the transformer backbone

In [ ]:
#### YOUR CODE STARTS HERE ####
backbone = ...

x = torch.randn(8, 3, 224, 224)
out = backbone(x)

# print the feature shapes
...
#### YOUR CODE ENDS HERE ####

<font color='red'> **Q1:** What do you notice about the **dimension ordering** for ResNet and Swin models? Describe it using `{b, c, h, w}` notation.  

<font color='red'> **Your answer goes here (Q1):**

...

----

<font color='red'> **Q2:** Can you explain the **resolution** of the first feature maps extracted from the ResNet and Swin models? Why do they differ, and what does this mean for the transformer backbone?  

<font color='red'> **Your answer goes here (Q2):**

...

<font color='red'>**(Homework exercise 2 – b)** To efficiently manage different backbone architectures, we will define **model wrappers** that:  
<font color='red'>
- Load models from the `timm` library  
- Store the `embed_dims` corresponding to each model  
- Provide an easy interface for accessing feature maps  
</font>

This will allow us to quickly swap between different architectures while keeping track of their feature extraction properties. Create you model collection for an easy access. (1 points)</font>

<font color='red'> Now, let's define model wrappers for easy creation and access to our model zoo.  

<font color='red'> **Tip:** Refer back to **Q1** to correctly reshape the Swin Transformer dimensions!  

In [ ]:
class ResNet50(nn.Module):
	def __init__(self, pretrained=True):
		super().__init__()
        #### YOUR CODE STARTS HERE ####
		self.backbone = ...
		self.embed_dims = [...]
        #### YOUR CODE ENDS HERE ####

	def forward(self, x):
		x = self.backbone(x)
		return x

In [ ]:
class ResNet101(nn.Module):
	def __init__(self, pretrained=True):
		super().__init__()
        #### YOUR CODE STARTS HERE ####
		self.backbone = ...
		self.embed_dims = [...]
        #### YOUR CODE ENDS HERE ####

	def forward(self, x):
		x = self.backbone(x)
		return x

In [ ]:
class SwinTransformer_t(nn.Module):
	def __init__(self, img_size=224, pretrained=True):
		super().__init__()
        #### YOUR CODE STARTS HERE ####
		self.backbone = ...
		self.embed_dims = [...]
        #### YOUR CODE ENDS HERE ####

	def forward(self, x):
        #### YOUR CODE STARTS HERE ####
		x = self.backbone(x)
		x = ... # correctly reshape dimensions
        #### YOUR CODE ENDS HERE ####
		return x

In [ ]:
class SwinTransformer_s(nn.Module):
	def __init__(self, img_size=224, pretrained=True):
		super().__init__()
        #### YOUR CODE STARTS HERE ####
		self.backbone = ...
		self.embed_dims = [...]
        #### YOUR CODE ENDS HERE ####

	def forward(self, x):
        #### YOUR CODE STARTS HERE ####
		x = self.backbone(x)
		x = ... # correctly reshape dimensions
        #### YOUR CODE ENDS HERE ####
		return x

In [ ]:
class SwinTransformer_b(nn.Module):
	def __init__(self, img_size=224, pretrained=True):
		super().__init__()
        #### YOUR CODE STARTS HERE ####
		self.backbone = ...
		self.embed_dims = [...]
        #### YOUR CODE ENDS HERE ####

	def forward(self, x):
        #### YOUR CODE STARTS HERE ####
		x = self.backbone(x)
		x = ... # correctly reshape dimensions
        #### YOUR CODE ENDS HERE ####
		return x

In [ ]:
resnet50 = ResNet50()
resnet50

<font color='red'> **Q:** What can you say about the model architecture?  
- <font color='red'>How many layers does it have?  
- What is the output size of the last layer?  
- How many parameters does the model have?  

<font color='red'> Use `model.parameters()` to count the total number of trainable parameters.

<font color='red'> **Your answer goes here:**

...

<font color='red'> Now, let's compare some of our convolutional and transformer-based models.  

<font color='red'> For a fair comparison, we need to **match models with similar parameter counts**.  

<font color='red'> **Q:** What is the equivalent Swin model in terms of parameters to **ResNet-50**? How about **ResNet-101**?  

<font color='red'> Write code to:
1. <font color='red'> Get the number of parameters for each model.
2. Pick the best equivalent **Swin version** for each ResNet.
3. Report the number of parameters for each model in **millions** (e.g., `00.00 M`).
4. **Do not remove unsuccessful attempts**—report all your comparisons :)

In [ ]:
#### YOUR CODE STARTS HERE ####
resnet50 = ResNet50()
params = ...

# get correct swin model
...
#### YOUR CODE ENDS HERE ####

In [ ]:
#### YOUR CODE STARTS HERE ####
resnet101 = ...

# get correct swin model
...
#### YOUR CODE ENDS HERE ####

<font color='red'>**(Homework exercise 2 – c)** Implement **Feature Pyramid Network** (FPN) following the instructions below. (1 points)</font>

<font color='red'> Now, let's define our **decoder**.  

<font color='red'> We will use the **same convolutional decoder architecture** as we did in the second practice session. Use the **simple FPN decoder** and correctly extract the `embed_dims`, which correspond to the dimensions of our multi-scale encoder feature maps.  

<font color='red'> Make sure you use the **same FPN architecture** as in the second practice session!  



In [ ]:
class FPN(nn.Module):
    #### YOUR CODE STARTS HERE ####
	...
    #### YOUR CODE ENDS HERE ####

In [ ]:
x = torch.randn(8, 3, 256, 256)

resnet50_fpn = FPN(resnet50, num_classes=1)
out = resnet50_fpn(x)
out.shape

In [ ]:
# here is a sanity check for an output shape
assert out.shape == torch.Size([8, 1, 256, 256])

<font color='red'>**(Homework exercise 2 – d)** Implement some meaningful tranformations. (1 points)</font>

<font color='red'> Similar to our practice session we will be using the [VOCSegmentation](http://host.robots.ox.ac.uk/pascal/VOC/voc2007/) dataset.

In [ ]:
train_dataset = VOCSegmentation(root="./data", year="2012", image_set="train", download=True)
valid_dataset = VOCSegmentation(root="./data", year="2012", image_set="val", download=True)

<font color='red'> Implement your transforms. You can read more about Albumentations here (https://albumentations.ai/docs/).

<font color='red'> For more specific examples follow: https://albumentations.ai/docs/examples/example_kaggle_salt/

In [ ]:
#### YOUR CODE STARTS HERE ####
train_transforms = A.Compose([
	A.Resize(...),
	A.HorizontalFlip(p=0.5),
	A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

valid_transforms = A.Compose([
	...
])
#### YOUR CODE ENDS HERE ####

<font color='red'> The dataset below will **filter out images** and leave only images and corresponding annotations of a certain class label (e.g., `target_class=3`). Thus, we will be performing a **binary segmentation task**.  

In [ ]:
class VOCBinaryDataset(Dataset):
	def __init__(self, dataset, transforms=None, target_class=3):
		self.dataset = dataset
		self.transforms = transforms
		self.target_class = target_class
		self.filtered_indices = self.filter_dataset()

	def filter_dataset(self):
		filtered_indices = []
		for idx in range(len(self.dataset)):
			_, mask = self.dataset[idx]
			mask = np.array(mask)
			if (mask == self.target_class).any():
				filtered_indices.append(idx)
		return filtered_indices

	def __len__(self):
		return len(self.filtered_indices)

	def __getitem__(self, idx):
		original_idx = self.filtered_indices[idx]
		image, mask = self.dataset[original_idx]
		image = np.array(image)
		mask = np.array(mask)

		mask = np.where(mask == self.target_class, 1, 0)

		if self.transforms:
			data = self.transforms(image=image, mask=mask)
			image = data['image']
			mask = data['mask']

		mask = np.expand_dims(mask, axis=0)
		image = torch.tensor(image.transpose(2, 0, 1)).float()
		mask = torch.tensor(mask).float()

		return image, mask


_train_dataset = VOCBinaryDataset(train_dataset, transforms=train_transforms)
_valid_dataset = VOCBinaryDataset(valid_dataset, transforms=valid_transforms)

train_loader = DataLoader(_train_dataset, batch_size=4, shuffle=True, num_workers=2)
val_loader = DataLoader(_valid_dataset, batch_size=4, shuffle=False, num_workers=2)

In [ ]:
def show_sample_images(dataset, num_samples=2):
    fig, axes = plt.subplots(num_samples, 2, figsize=(4, num_samples * 2))

    for i in range(num_samples):
        img, mask = dataset[i]

        img = img.permute(1, 2, 0).numpy()
        img = (img * np.array([0.229, 0.224, 0.225])) + np.array([0.485, 0.456, 0.406])
        mask = mask.numpy()

        axes[i, 0].imshow(img)
        axes[i, 0].set_title("Image")
        axes[i, 0].axis("off")

        axes[i, 1].imshow(mask[0])
        axes[i, 1].set_title("Mask")
        axes[i, 1].axis("off")

    plt.tight_layout()
    plt.show()

show_sample_images(_train_dataset, num_samples=2)

In [ ]:
def compute_mean_iou(pred, target):
	pred = (pred > 0.5).float()
	intersection = (pred * target).sum(dim=(1, 2, 3))
	union = (pred + target).sum(dim=(1, 2, 3)) - intersection
	iou = (intersection + 1e-6) / (union + 1e-6)
	return iou.mean().item()

<font color='red'>**(Homework exercise 2 – e)** As we did in the previous exercise, implement a SegTrainer for handling the training of segmentation models. (2 points)</font>

<font color='red'> Just like for classification, we can implement a **general pipeline Trainer** for training our models and evaluating them on our dataset.  

<font color='red'> Below, fill in the needed parts. You should use `compute_mean_iou()` to compute the running IoU scores.  

In [ ]:
class SegTrainer:
    def __init__(self,
                 max_epochs,
                 model,
                 criterion,
                 optimizer,
                 scheduler,
                 device,
                 train_loader,
                 valid_loader,
                 test_loader,
                 log_every_n_steps=5
    ) -> None:
        self.model = model
        self.criterion = criterion
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.device = device
        self.train_loader = train_loader
        self.valid_loader = valid_loader
        self.test_loader = test_loader
        self.max_epochs = max_epochs
        self.log_every_n_steps = log_every_n_steps


    def train_step(self):
        self.model.train()
        running_loss = 0.0
        metric = 0.0
        total = 0

        pbar = tqdm(self.train_loader, desc="Training")

        for batch_idx, (inputs, targets) in enumerate(pbar):
            inputs, targets = inputs.to(self.device), targets.to(self.device)

            #### YOUR CODE STARTS HERE ####
            ...
            #### YOUR CODE ENDS HERE ####

            running_loss += loss.item()

            #### YOUR CODE STARTS HERE ####
            metric += ...
            total += targets.size(0)

            if batch_idx % self.log_every_n_steps == 0:
                avg_loss = ...
                iou = ...
                pbar.set_postfix(loss=f"{avg_loss:.4f}", iou=f"{iou:.4f}")
            #### YOUR CODE ENDS HERE ####

        avg_loss = running_loss / len(self.train_loader)
        iou = metric / len(self.train_loader)
        pbar.set_postfix(loss=f"{avg_loss:.4f}", iou=f"{iou:.4f}")
        return avg_loss, iou


    def valid_step(self):
        self.model.eval()
        running_loss = 0.0
        metric = 0.0
        total = 0

        with torch.no_grad():
            pbar = tqdm(self.valid_loader, desc="Validation")

            for batch_idx, (inputs, targets) in enumerate(pbar):
                inputs, targets = inputs.to(self.device), targets.to(self.device)

                #### YOUR CODE STARTS HERE ####
                ...
                #### YOUR CODE ENDS HERE ####

                running_loss += loss.item()

                #### YOUR CODE STARTS HERE ####
                metric += ...
                total += targets.size(0)

                if batch_idx % self.log_every_n_steps == 0:
                    avg_loss = ...
                    iou = ...
                    pbar.set_postfix(loss=f"{avg_loss:.4f}", iou=f"{iou:.4f}")
                #### YOUR CODE ENDS HERE ####

            avg_loss = running_loss / len(self.valid_loader)
            iou = metric / len(self.valid_loader)
            pbar.set_postfix(loss=f"{avg_loss:.4f}", iou=f"{iou:.4f}")

        return avg_loss, iou


    def train(self):
        self.model.to(self.device)
        train_losses = []
        train_ious = []
        valid_losses = []
        valid_ious = []

        for epoch in range(self.max_epochs):
            train_loss, train_iou = self.train_step()
            valid_loss, valid_iou = self.valid_step()

            self.scheduler.step()

            train_losses.append(train_loss)
            train_ious.append(train_iou)
            valid_losses.append(valid_loss)
            valid_ious.append(valid_iou)

            print(f"Epoch {epoch+1}/{self.max_epochs}: train_loss: {train_loss:.4f}, train_iou: {train_iou:.4f}, valid_loss: {valid_loss:.4f}, valid_iou: {valid_iou:.4f}")
            print()

        return train_losses, train_ious, valid_losses, valid_ious


    def test(self):
        self.model.eval()
        metric = 0.0
        total = 0

        with torch.no_grad():
            pbar = tqdm(self.test_loader, desc="Testing")

            for inputs, targets in pbar:
                #### YOUR CODE STARTS HERE ####
                ...

                metric += ...
                total += targets.size(0)
                #### YOUR CODE ENDS HERE ####

            iou = metric / len(self.test_loader)
            print(f"test_iou: {iou:.4f}")

        return iou


    def plot_metrics(self, train_losses, train_ious, valid_losses, valid_ious):
        fig, ax = plt.subplots(1, 2, figsize=(15, 5))
        ax[0].plot(train_losses, label="train_loss", color="blue")
        ax[0].plot(valid_losses, label="valid_loss", linestyle="--", color="orange")
        ax[0].set_title("Losses")

        ax[1].plot(train_ious, label="train_iou", color="blue")
        ax[1].plot(valid_ious, label="valid_iou", linestyle="--", color="orange")
        ax[1].set_title("IoU")

        for i in range(2):
            ax[i].legend()
            ax[i].grid(0.35)
            ax[i].set_xlabel("Epochs")

        plt.show()

In [ ]:
def plot_segm_preds(model, dataloader, num_preds=20):
	model.eval()
	shown = 0

	with torch.no_grad():
		for images, masks in dataloader:
			images, masks = images.to(device), masks.to(device)
			outputs = model(images)
			predictions = outputs.sigmoid()

			batch_size = images.shape[0]
			n = min(num_preds - shown, batch_size)

			for i in range(n):
				image = images[i].permute(1, 2, 0).cpu().numpy()
				image = image * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
				image = np.clip(image, 0, 1)

				mask = masks[i].cpu().numpy()[0]
				pred = predictions[i].cpu().numpy()[0]

				fig, axes = plt.subplots(1, 3, figsize=(12, 4))
				for ax, img, title in zip(axes, [image, mask, pred], ["Image", "Mask", "Prediction"]):
					ax.imshow(img)
					ax.set_title(title)
					ax.axis("off")
				plt.tight_layout()
				plt.show()

				shown += 1
				if shown >= num_preds:
					break

<font color='red'>**(Homework exercise 2 – f)** Conduct your own model benchmarking, report and reason the performance of all models. (2 points)</font>

<font color='red'> For this homework, we will **test different encoder architectures** with the **same FPN decoder** for a segmentation task.  

<font color='red'> We need to compare how different the performance of convolutional ResNet-50 and ResNet-101 is compared to their **corresponding Swin backbones**.  

<font color='red'> We will store our benchmark results in a dictionary as follows:  

```python
benchmark_results['resnet50_fpn'] = {
    'train_losses': train_losses,
    'train_ious': train_ious,
    'valid_losses': valid_losses,
    'valid_ious': valid_ious
}
```


<font color='red'> For each new experiment, don’t forget to:
- <font color='red'> Initialize the correct model wrapper class that loads a pretrained version of that model using the `timm` library.
- Pass this backbone as an argument to the `FPN` class with the correct number of classes for the segmentation task.
- Define your model-specific parameters (**Tip:** Watch out for how you define parameters for your optimizer).
- Use the `SegTrainer` with enough epochs to train your models.

<font color='red'> When training the models below you should expect to get IoU scores **>0.6**.

In [ ]:
benchmark_results = {}

### ResNet50

In [ ]:
#### YOUR CODE STARTS HERE ####
resnet50 = ...
resnet50_fpn = FPN(
    backbone=...
	num_classes=...
).to(device)
#### YOUR CODE ENDS HERE ####

In [ ]:
#### YOUR CODE STARTS HERE ####
optimizer = ...
criterion = ...
scheduler = ...


trainer = SegTrainer(
	max_epochs=...,
	model=resnet50_fpn,
	criterion=criterion,
	optimizer=optimizer,
	scheduler=scheduler,
	device=device,
	train_loader=train_loader,
	valid_loader=val_loader,
	test_loader=val_loader
)
#### YOUR CODE ENDS HERE ####

In [ ]:
train_losses, train_ious, valid_losses, valid_ious = trainer.train()

In [ ]:
benchmark_results['resnet50_fpn'] = {
	'train_losses': train_losses,
	'train_ious': train_ious,
	'valid_losses': valid_losses,
	'valid_ious': valid_ious
}

In [ ]:
trainer.plot_metrics(train_losses, train_ious, valid_losses, valid_ious)

In [ ]:
trainer.test()

In [ ]:
plot_segm_preds(resnet50_fpn, val_loader, num_preds=5)

### ResNet101

In [ ]:
#### YOUR CODE STARTS HERE ####
resnet101 = ...
resnet101_fpn = FPN(
	backbone=...,
	num_classes=...
).to(device)
#### YOUR CODE ENDS HERE ####

In [ ]:
#### YOUR CODE STARTS HERE ####
optimizer = ...
criterion = ...
scheduler = ...


trainer = SegTrainer(
	max_epochs=...,
	model=resnet101_fpn,
	criterion=criterion,
	optimizer=optimizer,
	scheduler=scheduler,
	device=device,
	train_loader=train_loader,
	valid_loader=val_loader,
	test_loader=val_loader
)
#### YOUR CODE ENDS HERE ####

In [ ]:
train_losses, train_ious, valid_losses, valid_ious = trainer.train()

In [ ]:
benchmark_results['resnet101_fpn'] = {
	'train_losses': train_losses,
	'train_ious': train_ious,
	'valid_losses': valid_losses,
	'valid_ious': valid_ious
}

In [ ]:
trainer.plot_metrics(train_losses, train_ious, valid_losses, valid_ious)

In [ ]:
trainer.test()

In [ ]:
plot_segm_preds(resnet101_fpn, val_loader, num_preds=5)

### SWIN-{your choice no.1} -- **fill this in**

In [ ]:
#### YOUR CODE STARTS HERE ####
swin_<> = ...
swin_<>_fpn = FPN(
	backbone=...,
	num_classes=...
).to(device)
#### YOUR CODE ENDS HERE ####

In [ ]:
#### YOUR CODE STARTS HERE ####
optimizer = ...
criterion = ...
scheduler = ...


trainer = SegTrainer(
	max_epochs=...,
	model=swin_<>_fpn,
	criterion=criterion,
	optimizer=optimizer,
	scheduler=scheduler,
	device=device,
	train_loader=train_loader,
	valid_loader=val_loader,
	test_loader=val_loader
)
#### YOUR CODE ENDS HERE ####

In [ ]:
train_losses, train_ious, valid_losses, valid_ious = trainer.train()

In [ ]:
benchmark_results['...'] = {
	'train_losses': train_losses,
	'train_ious': train_ious,
	'valid_losses': valid_losses,
	'valid_ious': valid_ious
}

In [ ]:
trainer.plot_metrics(train_losses, train_ious, valid_losses, valid_ious)

In [ ]:
trainer.test()

In [ ]:
plot_segm_preds(... val_loader, num_preds=5)

### SWIN-{your choice no.2} -- fill this in

In [ ]:
#### YOUR CODE STARTS HERE ####
swin_<> = ...
swin_<>_fpn = FPN(
	backbone=...,
	num_classes=...
).to(device)
#### YOUR CODE ENDS HERE ####

In [ ]:
#### YOUR CODE STARTS HERE ####
optimizer = ...
criterion = ...
scheduler = ...


trainer = SegTrainer(
	max_epochs=...,
	model=swin_<>_fpn,
	criterion=criterion,
	optimizer=optimizer,
	scheduler=scheduler,
	device=device,
	train_loader=train_loader,
	valid_loader=val_loader,
	test_loader=val_loader
)
#### YOUR CODE ENDS HERE ####

In [ ]:
train_losses, train_ious, valid_losses, valid_ious = trainer.train()

In [ ]:
benchmark_results['...'] = {
	'train_losses': train_losses,
	'train_ious': train_ious,
	'valid_losses': valid_losses,
	'valid_ious': valid_ious
}

In [ ]:
trainer.plot_metrics(train_losses, train_ious, valid_losses, valid_ious)

In [ ]:
trainer.test()

In [ ]:
plot_segm_preds(..., val_loader, num_preds=5)

In [ ]:
# given benchmark results plot the metrics for each model

fig, ax = plt.subplots(2, 2, figsize=(15, 10))
for model, metrics in benchmark_results.items():
	if not metrics:
		continue
	ax[0, 0].plot(metrics['train_losses'], label=f"{model}", linewidth=2)
	ax[0, 1].plot(metrics['train_ious'], label=f"{model}", linewidth=2)

	ax[1, 0].plot(metrics['valid_losses'], label=f"{model}", linewidth=2)
	ax[1, 1].plot(metrics['valid_ious'], label=f"{model}", linewidth=2)

ax[0, 0].set_title("Train Losses")
ax[0, 1].set_title("Train IoU")
ax[1, 0].set_title("Valid Losses")
ax[1, 1].set_title("Valid IoU")

for i in range(2):
	for j in range(2):
		ax[i, j].legend()
		ax[i, j].grid(0.35)
		ax[i, j].set_xlabel("Epochs")

plt.tight_layout()
plt.show()

<font color='red'> Describe your interpretation of the results from the plots above. What worked better, and why do you think this is the case? What configurations did you try for your models, and how did they influence the final performance? How about learning rates? What worked better?  

<font color='red'> You can find some tips and tricks on how to train your transformer here (https://arxiv.org/abs/1804.00247).

<font color='red'> **Your answer goes here:**

...

# **Exercise 2 Bonus (3 points): Top 3 on Leaderboard**

## <font color='red'> **Bonus Task: Optimizing Segmentation IoU with Different Backbones**  

<font color='red'> In this task, you must **experiment with different backbone models** to achieve the highest **Intersection over Union (IoU) score** on the validation dataset provided above. The goal is to explore different architectures and training strategies while maintaining a structured and well-documented approach.  

### <font color='red'> **Rules and Constraints**  

- <font color='red'> **Backbone Selection:** You may use any pretrained backbone from `timm`, implement a custom architecture, or combine multiple models. The backbone must be integrated into a **segmentation model** to predict masks. Ensure that the backbone is properly adapted for multi-scale feature extraction if needed.  

- <font color='red'>**Decoder Requirement:** The decoder architecture must be **the same FPN decoder used in the second practice session** to ensure fair comparisons. You may tune hyperparameters of the decoder, but structural changes are not allowed.  

- <font color='red'>**Evaluation and Dataset:** IoU scores must be evaluated **exclusively on the validation set** provided above. Ensure that all results are obtained **from the same dataset split** to maintain consistency.  

- <font color='red'>**Code Submission:** Write your **entire implementation** in the designated code cell below. The code must be **runnable and reproducible**. Clearly indicate **which backbone model** you used.  

- <font color='red'>**Result Reporting:** A **detailed report is mandatory** for obtaining points. The report must include: model configurations (backbone selection, parameter choices), training details (learning rate, optimizer, batch size, augmentation techniques), performance comparison (what worked, what did not work, final results), and any additional insights from the experiments.  

### <font color='red'> **Scoring Criteria (On Validation set provided above)**  

- <font color='red'> **3 points** -> Awarded to the **best overall IoU score** (highest IoU score).  
- <font color='red'> **2 points** -> Awarded to the **next three highest IoU scores**.  
- <font color='red'> **1 point** -> Awarded to the **next five highest IoU scores**.  

### <font color='red'> **Strict Grading Policy**  

- <font color='red'> **The report is mandatory.** If your model achieves a high IoU score but you do not submit a detailed report, you **will not receive bonus points**.  
- <font color='red'> IoU scores must be **valid and reproducible**. Any inconsistencies or lack of transparency in reporting will result in 0 bonus points.  

### <font color='red'> **Instructions for Submission**  

- <font color='red'> **Implement your model in the designated code cell.**  
- <font color='red'> **Train and evaluate** the model using the validation dataset.  
- <font color='red'> **Report your best IoU score along with a detailed analysis.**  

<font color='red'> Failure to comply with the above requirements may result in disqualification from the bonus points.

<font color='red'> Good luck! :)

In [ ]:
#### YOUR CODE STARTS HERE ####
...
#### YOUR CODE ENDS HERE ####

In [ ]:
#### YOUR REPORT STARTS HERE ####
# provide clear solution and report your best iou score here (at the end)
...
#### YOUR REPORT ENDS HERE ####

# **Exercise 3 (5 points): HuggingFace**

<font color='red'>We will work with HuggingFace once again, this time we will be using DETR for object detection https://arxiv.org/abs/2005.12872

We have separate imports for this exercise, so that you can work on it without touching the previous ones.</font>

In [ ]:
# Python 3.12.13 on Colab
!python --version

In [ ]:
!pip install -q transformers[torch] datasets

In [ ]:
!pip install --upgrade transformers datasets huggingface_hub

In [ ]:
import transformers
import datasets
print(transformers.__version__) # 5.5.1
print(datasets.__version__) # 4.8.4

In [ ]:
from datasets import load_dataset
from transformers import AutoImageProcessor, DetrImageProcessor
from transformers import AutoModelForObjectDetection
from transformers import Trainer, TrainingArguments
import matplotlib.patches as patches

<font color='red'>To work with HuggingFace and havethe ability to upload models there, you need to have an account on HuggingFace. Log in to your account and generate a new token with permission to **write**.

To upload models to Hugging Face, you need an account.  

1. **Log in** at [huggingface.co](https://huggingface.co/) or create an account.  
2. **Generate a token**: Go to **Settings → Access Tokens → New Token**, name it, set permissions to **Write**, and generate it.  
3. **Copy the token** (it won’t be shown again).
4. **Paste it in the box below**

Now you can upload models to Hugging Face.</font>

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()

from huggingface_hub import login
login(token="hf_XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")

<font color='red'>We will work with fashionpedia dataset https://huggingface.co/datasets/detection-datasets/fashionpedia_4_categories</font>

In [ ]:
dataset = load_dataset('detection-datasets/fashionpedia_4_categories')

<font color='red'>First lets choose a reasonable amount of trainin data. The recomended size for train set is >1000 samples.</font>

In [ ]:
#### YOUR CODE STARTS HERE ####
train_size = ...
valid_size = ...
test_size = ...

train_indices = np.random.choice(len(dataset['train']), size=train_size, replace=False)
train_dataset = dataset['train'].select(train_indices)
valid_indices = np.random.choice(len(dataset['val']), size=valid_size, replace=False)
valid_dataset = dataset['val'].select(valid_indices)
test_indices = np.random.choice(len(dataset['test']), size=test_size, replace=False)
test_dataset = dataset['test'].select(test_indices)
#### YOUR CODE ENDS HERE ####

In [ ]:
len(train_dataset), len(valid_dataset), len(test_dataset)

<font color='red'>**(Homework exercise 3 – a)** Inspect how the detection dataset is structured. Pick one example from `train_dataset`, print the image size and its object annotations, and recover the class names from the dataset features. Pay attention to the available keys inside `objects` and to the bounding-box format used by the dataset. (1 point)</font>

In [ ]:
#### YOUR CODE STARTS HERE ####
# check a sample.
sample = ...
image = ...
annotations = ...
print(f"Sample image size: {...}, Objects: {...}")
#### YOUR CODE ENDS HERE ####

In [ ]:
#### YOUR CODE STARTS HERE ####
# get categories for the dataset
categories = ...
categories[:10], len(categories)
#### YOUR CODE ENDS HERE ####

In [ ]:
fig, ax = plt.subplots(1, figsize=(4, 4))
ax.imshow(image)

bboxes = annotations['bbox']
labels = annotations['category']

for bbox, label in zip(bboxes, labels):
	x1, y1, x2, y2 = bbox
	rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor='r', facecolor='none')
	ax.add_patch(rect)
	ax.text(x1, y1, categories[label], fontsize=8, color='white', bbox=dict(facecolor='red', alpha=0.5))

plt.tight_layout()
plt.axis("off")
plt.show()

<font color='red'>Now we will load a pretrained object detection model from the Hugging Face Hub. You can start with a DETR-family checkpoint such as `microsoft/conditional-detr-resnet-50` or `facebook/detr-resnet-50`, but feel free to experiment with other object detection checkpoints that work with `AutoImageProcessor` and `AutoModelForObjectDetection`. Browse current object detection models here: https://huggingface.co/models?pipeline_tag=object-detection&sort=downloads. Whenever you change the checkpoint, load the matching image processor from the same repo.</font>

In [ ]:
checkpoint = "facebook/detr-resnet-50"
image_size = 480

image_processor = AutoImageProcessor.from_pretrained(
    checkpoint,
)

In [ ]:
image_processor

<font color='red'>**(Homework exercise 3 – b)** Prepare the dataset for training. Load the image processor for your chosen checkpoint, define `albumentations` augmentations, and convert annotations into the format expected by the model. Good references are the Hugging Face object detection guide https://huggingface.co/docs/transformers/en/tasks/object_detection and https://huggingface.co/learn/cookbook/fine_tuning_detr_custom_dataset#7-add-data-augmentation-to-the-dataset, the Albumentations bounding-box guide https://albumentations.ai/docs/3-basic-usage/bounding-boxes-augmentations/, and the Albumentations transform explorer https://explore.albumentations.ai/. (1 point)</font>

In [ ]:
#### YOUR CODE STARTS HERE ####
train_transform = A.Compose(
    [
        ...
    ],
    bbox_params=...
)

validation_transform = A.Compose(
    [
        ...
    ],
    bbox_params=...
)
#### YOUR CODE ENDS HERE ####

<font color='red'>The following helper functions sanitize bounding boxes after augmentation, convert them to COCO-style detection annotations, and package each example in the format expected by the image processor.</font>

In [ ]:
def sanitize_boxes_and_labels(bboxes, categories, image_shape):
    height, width = image_shape[:2]
    sanitized_bboxes = []
    sanitized_categories = []

    for bbox, category_id in zip(bboxes, categories):
        x1, y1, x2, y2 = [float(value) for value in bbox]

        x1, x2 = sorted((x1, x2))
        y1, y2 = sorted((y1, y2))

        x1 = min(max(x1, 0.0), width - 1.0)
        y1 = min(max(y1, 0.0), height - 1.0)
        x2 = min(max(x2, 0.0), width - 1.0)
        y2 = min(max(y2, 0.0), height - 1.0)

        if (x2 - x1) >= 1.0 and (y2 - y1) >= 1.0:
            sanitized_bboxes.append([x1, y1, x2, y2])
            sanitized_categories.append(int(category_id))

    return sanitized_bboxes, sanitized_categories


def convert_pascal_voc_to_coco(bbox):
    xmin, ymin, xmax, ymax = bbox
    width = xmax - xmin
    height = ymax - ymin
    return [xmin, ymin, width, height]


def format_image_annotations_as_coco(image_id, categories, bboxes):
    annotations = []
    for category_id, bbox in zip(categories, bboxes):
        coco_bbox = convert_pascal_voc_to_coco(bbox)
        _, _, width, height = coco_bbox
        annotations.append(
            {
                "image_id": image_id,
                "category_id": category_id,
                "iscrowd": 0,
                "area": float(width * height),
                "bbox": coco_bbox,
            }
        )
    return {"image_id": image_id, "annotations": annotations}


def custom_transforms(examples, transform):
    images = []
    annotations = []

    for image_id, image, objects in zip(examples["image_id"], examples["image"], examples["objects"]):
        image = np.array(image.convert("RGB"))
        output = transform(image=image, bboxes=objects["bbox"], category=objects["category"])
        bboxes, categories = sanitize_boxes_and_labels(output["bboxes"], output["category"], output["image"].shape)

        images.append(output["image"])
        annotations.append(
            format_image_annotations_as_coco(
                image_id=image_id,
                categories=categories,
                bboxes=bboxes,
            )
        )

    return image_processor(images=images, annotations=annotations, return_tensors="pt")

<font color='red'>Let's transform the dataset and inspect one processed training sample. This is a good place to verify that resizing and augmentation still leave the boxes aligned with the objects and that the labels remain correct.</font>

In [ ]:
_train_dataset = train_dataset.with_transform(lambda x: custom_transforms(x, train_transform))
_valid_dataset = valid_dataset.with_transform(lambda x: custom_transforms(x, validation_transform))
_test_dataset = test_dataset.with_transform(lambda x: custom_transforms(x, validation_transform))

In [ ]:
from transformers.image_transforms import center_to_corners_format


# check a sample
sample = _train_dataset[25]
pixel_values = sample["pixel_values"]
if pixel_values.ndim == 4:
    pixel_values = pixel_values.squeeze(0)

image = pixel_values.permute(1, 2, 0).numpy()
image = image * np.array(image_processor.image_std) + np.array(image_processor.image_mean)
image = np.clip(image, 0.0, 1.0)

fig, ax = plt.subplots(1, figsize=(5, 5))
ax.imshow(image)

annotations = sample["labels"]
bboxes = center_to_corners_format(annotations["boxes"]).clone()
height, width = image.shape[:2]
scale = torch.tensor([width, height, width, height], dtype=bboxes.dtype)
bboxes = bboxes * scale

labels = annotations["class_labels"]
for bbox, label in zip(bboxes, labels):
    x1, y1, x2, y2 = bbox.tolist()
    rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor="r", facecolor="none")
    ax.add_patch(rect)
    ax.text(
        x1,
        y1,
        categories[int(label)],
        fontsize=8,
        color="white",
        bbox=dict(facecolor="red", alpha=0.5),
    )

plt.tight_layout()
plt.axis("off")
plt.show()

In [ ]:
image.shape

<font color='red'>**(Homework exercise 3 – c)** Train your model. You may experiment with the training arguments, but keep the run realistic enough to finish and compare. For up-to-date guidance, consult the current `Trainer` docs https://huggingface.co/docs/transformers/en/main_classes/trainer and the object detection task guide https://huggingface.co/docs/transformers/en/tasks/object_detection. If you switch to a different checkpoint, make sure the model, image processor, collator, and inference code stay consistent. (2 points)</font>

<font color='red'>We can pass `id2label` and `label2id` together with the model so saved checkpoints keep human-readable class names for evaluation, visualization, and inference.</font>

In [ ]:
#### YOUR CODE STARTS HERE ####
# dictionary: class id -> class name
id2label = ...
# dictionary: class name -> class id
label2id = ...
#### YOUR CODE ENDS HERE ####

In [ ]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained(
    checkpoint,
    id2label=...,
    label2id=...,
    num_labels=...,
)
model = AutoModelForObjectDetection.from_pretrained(
    checkpoint,
    config=config,
    ignore_mismatched_sizes=True,
)

In [ ]:
# note, be careful with the lr and wd for the detectrion transformer
# try using >= 5 epochs, experiemnts with warmups and lrs

#### YOUR CODE STARTS HERE ####
training_args = TrainingArguments(
    ...
)
#### YOUR CODE ENDS HERE ####

In [ ]:
def collate_fn(batch):
    pixel_values = [
        item["pixel_values"] if item["pixel_values"].ndim == 3 else item["pixel_values"].squeeze(0)
        for item in batch
    ]
    labels = [item["labels"] for item in batch]

    data = {
        "pixel_values": torch.stack(pixel_values),
        "labels": labels,
    }

    if "pixel_mask" in batch[0]:
        pixel_mask = [
            item["pixel_mask"] if item["pixel_mask"].ndim == 2 else item["pixel_mask"].squeeze(0)
            for item in batch
        ]
        data["pixel_mask"] = torch.stack(pixel_mask)

    return data

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=collate_fn,
    train_dataset=_train_dataset,
    eval_dataset=_valid_dataset,
    processing_class=image_processor,
)

trainer.train()

In [ ]:
try:
    from transformers.utils.notebook import NotebookProgressCallback
    trainer.remove_callback(NotebookProgressCallback)
except Exception:
    pass

In [ ]:
metrics = trainer.evaluate(eval_dataset=_test_dataset)
print(metrics)

<font color='red'>Push your trained model to the Hugging Face Hub so you can reuse it later for inference, sharing, or further fine-tuning.</font>

In [ ]:
trainer.push_to_hub()

<font color='red'>Add the link to your public model here if you pushed it to the Hub.

**Your answer goes here:**</font>

...

<font color='red'>**(Homework exercise 3 – d)** Run inference on an image of your choice from the test set. Use your checkpoint that you have just pushed to the Hub.

Preprocess the raw image with the matching `AutoImageProcessor`, choose a detection threshold, and visualize the predicted boxes and labels on the original image. (1 point)</font>

In [ ]:
#### YOUR CODE STARTS HERE ####
sample_idx = 6
sample = ...
image = ...

fig, ax = plt.subplots(1, figsize=(6, 6))
ax.imshow(image)
ax.axis("off")
plt.show()
#### YOUR CODE ENDS HERE ####

In [ ]:
#### YOUR CODE STARTS HERE ####
your_checkpoint = '<your hf checkpoint goes here>'

hf_image_processor = AutoImageProcessor.from_pretrained(your_checkpoint)
hf_model = AutoModelForObjectDetection.from_pretrained(your_checkpoint)
#### YOUR CODE ENDS HERE ####

In [ ]:
#### YOUR CODE STARTS HERE ####
# get predictions for your model

# prepare you model for inference
inference_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
hf_model = ...
...

# prepare inputs
inputs = hf_image_processor(...)
inputs = {key: value.to(inference_device) for key, value in inputs.items()}

with torch.no_grad():
    outputs = ...

# postprocess predictions
target_sizes = torch.tensor([image.size[::-1]], device=inference_device)
results = hf_image_processor.post_process_object_detection(
    outputs,
    threshold=...,
    target_sizes=target_sizes,
)[0]

if len(results["scores"]) == 0:
    print("No detections above the chosen threshold.")
else:
    for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
        box = [round(i, 2) for i in box.tolist()]
        print(
            f"Detected {hf_model.config.id2label[label.item()]} with confidence "
            f"{round(score.item(), 3)} at location {box}"
        )
#### YOUR CODE ENDS HERE ####

In [ ]:
fig, ax = plt.subplots(1, figsize=(6, 6))
ax.imshow(image)

for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
    x1, y1, x2, y2 = box.tolist()
    rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor="r", facecolor="none")
    ax.add_patch(rect)
    ax.text(
        x1,
        max(y1 - 5, 0),
        f"{hf_model.config.id2label[label.item()]}: {score.item():.2f}",
        fontsize=8,
        color="white",
        bbox=dict(facecolor="red", alpha=0.6),
    )

plt.tight_layout()
plt.axis("off")
plt.show()

# Comments (optional feedback to the course instructors)
Here, please, leave your comments regarding the homework, possibly answering the following questions:
* how much time did you send on this homework?
* was it too hard/easy for you?
* what would you suggest to add or remove?
* anything else you would like to tell us

Your comments:

# <font color='red'>  End of the homework. Please don't delete this cell.</font>